In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.base import BaseEstimator, ClassifierMixin
from catboost import CatBoostClassifier
import warnings

warnings.filterwarnings('error')
warnings.filterwarnings('ignore')
import pickle
import warnings

warnings.filterwarnings('ignore')
warnings.filterwarnings('error')

In [7]:
'''
Обертка для XGBClassifier, добавляющая метод __sklearn_tags__  
для совместимости со scikit-learn 1.7 и выше.  
Решает проблему с DeprecationWarning
'''


class SklearnXGBClassifier(xgb.XGBClassifier, BaseEstimator, ClassifierMixin):
    def __sklearn_tags__(self):
        return {
            'non_deterministic': True,
            'requires_fit': True,
            'X_types': ['2darray'],
        }

In [8]:
file_path = '../../../data/data_for_final_models/AgglomerativeClustering_generated_features.csv'
data = pd.read_csv(file_path)

X = data.drop(columns='Cluster')
y = data['Cluster']

kf = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'catboost': CatBoostClassifier(random_state=42, verbose=0),
    'lr': LogisticRegression(random_state=42, max_iter=10_000),
    'rf': RandomForestClassifier(random_state=42),
    'et': ExtraTreesClassifier(random_state=42),
    'gb': GradientBoostingClassifier(n_estimators=100, random_state=42)
}

meta_predictions = []
true_labels = []

for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    fold_predictions = {}
    
    # Обучение и предсказание для каждой модели
    for name, model in models.items():
        if name == 'catboost':
            model.fit(X_train, y_train, verbose=0)
        else:
            model.fit(X_train, y_train)
        
        preds = model.predict(X_val)
        # Обработка catboost предсказаний
        if name == 'catboost':
            preds = preds.ravel()
        fold_predictions[name] = preds
    
    # Создание мета-признаков
    X_meta = pd.DataFrame(fold_predictions)
    true_labels.extend(y_val)
    
    # Обучение мета-модели на текущем фолде
    meta_model = xgb.XGBClassifier(random_state=42, tree_method="auto")
    meta_model.fit(X_meta, y_val)
    
    # Сохранение предсказаний мета-модели
    meta_preds = meta_model.predict(X_meta)
    meta_predictions.extend(meta_preds)

In [9]:
final_accuracy = balanced_accuracy_score(true_labels, meta_predictions)
print(f'Mean Balanced Accuracy: {final_accuracy:.4f}')

Mean Balanced Accuracy: 0.9961


In [10]:
final_models = {}
for name, model in models.items():
    if name == 'catboost':
        model.fit(X, y, verbose=0)
    else:
        model.fit(X, y)
    final_models[name] = model

# Создание финальных мета-признаков
X_meta_final = pd.DataFrame({
    'catboost': final_models['catboost'].predict(X).ravel(),
    'lr': final_models['lr'].predict(X),
    'rf': final_models['rf'].predict(X),
    'et': final_models['et'].predict(X),
    'gb': final_models['gb'].predict(X)
})

# Обучение финальной мета-модели
final_meta_model = xgb.XGBClassifier(random_state=42, tree_method="auto")
final_meta_model.fit(X_meta_final, y)

DeprecationWarning: The XGBClassifier or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.

DeprecationWarning: The XGBClassifier or classes from which it inherits use `_get_tags` and `_more_tags`. Please define the `__sklearn_tags__` method, or inherit from `sklearn.base.BaseEstimator` and/or other appropriate mixins such as `sklearn.base.TransformerMixin`, `sklearn.base.ClassifierMixin`, `sklearn.base.RegressorMixin`, and `sklearn.base.OutlierMixin`. From scikit-learn 1.7, not defining `__sklearn_tags__` will raise an error.

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=None, n_jobs=None,
              num_parallel_tree=None, objective='multi:softprob', ...)

In [11]:
# Сохраняем модели

final_models['catboost'].save_model('../../../models/1_default_models/catboost_blend.cb')

for name in ['lr', 'rf', 'et', 'gb']:
    with open(f'../../../models/1_default_models/{name}_blend.pkl', 'wb') as f:
        pickle.dump(final_models[name], f)

with open('../../../models/1_default_models/meta_blend_model.pkl', 'wb') as f:
    pickle.dump(final_meta_model, f)